# 🔗 Link Prediction — Feature Engineering

Compute all 20 graph-based features for sampled train/test pairs and serialise them for `Model_Training.ipynb`.

### Pipeline
| Step | Notebook |
|------|----------|
| Graph construction & EDA | `EDA.ipynb` |
| **Centrality scores (Katz, HITS) + features** | **`Feature_eng.ipynb` ← you are here** |
| Model training & evaluation | `Model_Training.ipynb` |

### Sections
1. [Imports](#1-imports)
2. [Load Training Graph](#2-load-training-graph)
3. [Similarity Feature Functions](#3-similarity-feature-functions)
4. [Structural Feature Functions](#4-structural-feature-functions)
5. [Katz Centrality](#5-katz-centrality)
6. [HITS Scores](#6-hits-scores)
7. [Sample DataFrames](#7-sample-dataframes)
8. [Compute Feature Matrix](#8-compute-feature-matrix)


## 1. Imports

In [1]:
import os
import math
import pickle
import random
import warnings

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.notebook import tqdm
from collections import Counter

warnings.filterwarnings('ignore')
%matplotlib inline

print(' All libraries imported.')
print(f'   NetworkX : {nx.__version__}')
print(f'   pandas   : {pd.__version__}')

 All libraries imported.
   NetworkX : 3.6.1
   pandas   : 3.0.3


## 2. Load Training Graph

Only the **training** positive edges are used to build the graph — test edges must never leak into feature computation.

In [2]:
%%time

train_graph = nx.read_edgelist(
    'data/X_train_pos.csv',
    delimiter=',',
    create_using=nx.DiGraph(),
    nodetype=int,
)

print(f'Nodes : {train_graph.number_of_nodes():,}')
print(f'Edges : {train_graph.number_of_edges():,}')

Nodes : 1,780,564
Edges : 7,550,015
CPU times: total: 33.8 s
Wall time: 35 s


## 3. Similarity Feature Functions

In [3]:
#  Jaccard Similarity

def jaccard_for_followers(graph, a, b):
    """Jaccard similarity of the *follower* (predecessor) sets of a and b."""
    try:
        if not graph.has_node(a) or not graph.has_node(b):
            return 0
        pa, pb = set(graph.predecessors(a)), set(graph.predecessors(b))
        union = pa | pb
        return len(pa & pb) / len(union) if union else 0
    except ZeroDivisionError:
        return 0


def jaccard_for_followees(graph, a, b):
    """Jaccard similarity of the *followee* (successor) sets of a and b."""
    try:
        if not graph.has_node(a) or not graph.has_node(b):
            return 0
        sa, sb = set(graph.successors(a)), set(graph.successors(b))
        union = sa | sb
        return len(sa & sb) / len(union) if union else 0
    except ZeroDivisionError:
        return 0


# Cosine Distance 

def cosine_distance_for_followers(graph, a, b):
    """1 − cosine similarity of follower sets."""
    try:
        if not graph.has_node(a) or not graph.has_node(b):
            return 0
        pa, pb = set(graph.predecessors(a)), set(graph.predecessors(b))
        if not pa or not pb:
            return 1.0  # FIX: empty set → maximum distance, not 0
        return 1 - len(pa & pb) / (math.sqrt(len(pa)) * math.sqrt(len(pb)))
    except ZeroDivisionError:
        return 1.0


def cosine_distance_for_followees(graph, a, b):
    """1 − cosine similarity of followee sets."""
    try:
        if not graph.has_node(a) or not graph.has_node(b):
            return 0
        sa, sb = set(graph.successors(a)), set(graph.successors(b))
        if not sa or not sb:
            return 1.0  # FIX: empty set → maximum distance, not 0
        return 1 - len(sa & sb) / (math.sqrt(len(sa)) * math.sqrt(len(sb)))
    except ZeroDivisionError:
        return 1.0

print(' Similarity functions defined.')

 Similarity functions defined.


## 4. Structural Feature Functions

In [4]:
# Shortest Path 

def compute_shortest_path_length(graph, a, b):
    """
    Directed shortest path a → b.
    Temporarily removes the direct edge (if present) to avoid trivial length-1 answers.
    Returns -1 when no path exists.
    """
    edge_existed = graph.has_edge(a, b)
    try:
        if edge_existed:
            graph.remove_edge(a, b)
        length = nx.shortest_path_length(graph, source=a, target=b)
        if edge_existed:
            graph.add_edge(a, b)
        return length
    except (nx.NetworkXNoPath, nx.NodeNotFound):
        if edge_existed:
            graph.add_edge(a, b)
        return -1


# Same Weakly Connected Component 

def belongs_to_same_wcc(graph, wcc, node_to_wcc_id, a, b):
    """
    Binary WCC reachability feature. Returns 1 if b is reachable from a
    (or b already follows a), else 0. No -1 sentinel — consistent binary
    semantics for the classifier.

    FIX: old version returned -1 for different WCCs, introducing a phantom
    third class value that silently corrupted the feature column.
    FIX: uses pre-built node_to_wcc_id dict for O(1) component lookup
    instead of O(n_components) linear scan.
    """
    # Reverse edge already exists → reachable
    if graph.has_edge(b, a):
        return 1
    # Forward edge exists: check alternate path within same WCC
    if graph.has_edge(a, b):
        if node_to_wcc_id.get(a) == node_to_wcc_id.get(b):
            graph.remove_edge(a, b)
            result = 1 if compute_shortest_path_length(graph, a, b) != -1 else 0
            graph.add_edge(a, b)
            return result
        return 0
    # No direct edge: reachable iff in same WCC and path exists
    if node_to_wcc_id.get(a) != node_to_wcc_id.get(b):
        return 0
    return 1 if compute_shortest_path_length(graph, a, b) != -1 else 0


# Adamic-Adar Index

def calc_adar_in(graph, a, b):
    """Adamic-Adar using in-degree of common followees."""
    total = 0
    try:
        common = set(graph.successors(a)) & set(graph.successors(b))
        for node in common:
            in_deg = graph.in_degree(node)
            if in_deg > 1:
                total += 1 / math.log(in_deg)
    except Exception:
        pass
    return total


# Following Back 

def following_back(graph, a, b):
    """1 if b already follows a, else 0."""
    return 1 if graph.has_edge(b, a) else 0


print('Structural functions defined.')

Structural functions defined.


## 5. Katz Centrality

In [5]:
%%time

katz = nx.katz_centrality(train_graph, alpha=0.005, beta=1.0)

print(f'Min  : {min(katz.values()):.6f}')
print(f'Max  : {max(katz.values()):.6f}')
print(f'Mean : {sum(katz.values())/len(katz):.6f}')

with open('data/katz.pkl', 'wb') as f:
    pickle.dump(katz, f)
print('Saved data/katz.pkl')

Min  : 0.000731
Max  : 0.003417
Mean : 0.000748
Saved data/katz.pkl
CPU times: total: 2min 3s
Wall time: 2min 9s


## 6. HITS Scores

In [6]:
%%time

hits = nx.hits(train_graph, max_iter=1000, tol=1e-08, normalized=True)
hits_hub  = hits[0]
hits_auth = hits[1]

print(f'Hub  — min:{min(hits_hub.values()):.6f}  max:{max(hits_hub.values()):.6f}  mean:{sum(hits_hub.values())/len(hits_hub):.6f}')
print(f'Auth — min:{min(hits_auth.values()):.6f}  max:{max(hits_auth.values()):.6f}  mean:{sum(hits_auth.values())/len(hits_auth):.6f}')

with open('data/hits.pkl', 'wb') as f:
    pickle.dump(hits, f)
print('Saved data/hits.pkl')

Hub  — min:-0.000000  max:0.004900  mean:0.000001
Auth — min:-0.000000  max:0.005396  mean:0.000001
Saved data/hits.pkl
CPU times: total: 34.8 s
Wall time: 35 s


In [7]:
%%time
wcc = list(nx.weakly_connected_components(train_graph))
print(f'Weakly connected components: {len(wcc):,}')

# FIX: O(1) component lookup dict (avoids O(n_components) scan per pair)
node_to_wcc_id = {node: i for i, comp in enumerate(wcc) for node in comp}
print(f'node_to_wcc_id built — {len(node_to_wcc_id):,} nodes indexed')

Weakly connected components: 48,821
CPU times: total: 6.72 s
Wall time: 6.9 s


## 7. Sample DataFrames

`random.seed(42)` **must** be set once and not reset between train and test sampling — this guarantees the same rows are skipped as in `Model_Training.ipynb`.

In [8]:
%%time

random.seed(42)  # ← set ONCE; do not reset before test sampling

N_TRAIN_POS = 15_100_028
N_TRAIN_NEG = 15_100_028
N_TEST_POS  =  3_775_006
N_TEST_NEG  =  3_775_006

S_TRAIN = 100_000  # must match Model_Training.ipynb
S_TEST  =  25_000  # must match Model_Training.ipynb

skip_train_pos = sorted(random.sample(range(1, N_TRAIN_POS + 1), N_TRAIN_POS - S_TRAIN))
skip_train_neg = sorted(random.sample(range(1, N_TRAIN_NEG + 1), N_TRAIN_NEG - S_TRAIN))
skip_test_pos  = sorted(random.sample(range(1, N_TEST_POS  + 1), N_TEST_POS  - S_TEST))
skip_test_neg  = sorted(random.sample(range(1, N_TEST_NEG  + 1), N_TEST_NEG  - S_TEST))

print(f'Train samples — pos: {S_TRAIN:,}  neg: {S_TRAIN:,}  total: {S_TRAIN*2:,}')
print(f'Test  samples — pos: {S_TEST:,}   neg: {S_TEST:,}   total: {S_TEST*2:,}')

Train samples — pos: 100,000  neg: 100,000  total: 200,000
Test  samples — pos: 25,000   neg: 25,000   total: 50,000
CPU times: total: 34.9 s
Wall time: 36.2 s


In [9]:
%%time

COLS = ['source_node', 'destination_node']

df_train_pos = pd.read_csv('data/X_train_pos.csv', skiprows=skip_train_pos, names=COLS)
df_train_pos['indicator_link'] = 1

df_train_neg = pd.read_csv('data/X_train_neg.csv', skiprows=skip_train_neg, names=COLS)
df_train_neg['indicator_link'] = 0

df_final_train = (
    pd.concat([df_train_pos, df_train_neg], ignore_index=True)
      .sample(frac=1, random_state=42)
      .reset_index(drop=True)
)

df_test_pos = pd.read_csv('data/X_test_pos.csv', skiprows=skip_test_pos, names=COLS)
df_test_pos['indicator_link'] = 1

df_test_neg = pd.read_csv('data/X_test_neg.csv', skiprows=skip_test_neg, names=COLS)
df_test_neg['indicator_link'] = 0

df_final_test = (
    pd.concat([df_test_pos, df_test_neg], ignore_index=True)
      .sample(frac=1, random_state=42)
      .reset_index(drop=True)
)

print(f'df_final_train : {df_final_train.shape}')
print(df_final_train['indicator_link'].value_counts().to_string())
print()
print(f'df_final_test  : {df_final_test.shape}')
print(df_final_test['indicator_link'].value_counts().to_string())

df_final_train : (99838, 3)
indicator_link
0    49947
1    49891

df_final_test  : (25056, 3)
indicator_link
1    12537
0    12519
CPU times: total: 5.73 s
Wall time: 6.24 s


## 8. Compute Feature Matrix

In [10]:
def compute_feature_stage1(df, graph):
    """Follower/followee counts and intersection sizes."""
    num_followers_s, num_followees_s = [], []
    num_followers_d, num_followees_d = [], []
    inter_followers, inter_followees = [], []

    for _, row in tqdm(df.iterrows(), total=len(df), desc='Stage-1'):
        src, dst = row['source_node'], row['destination_node']
        s_in  = set(graph.predecessors(src)) if graph.has_node(src) else set()
        s_out = set(graph.successors(src))   if graph.has_node(src) else set()
        d_in  = set(graph.predecessors(dst)) if graph.has_node(dst) else set()
        d_out = set(graph.successors(dst))   if graph.has_node(dst) else set()

        num_followers_s.append(len(s_in));  num_followees_s.append(len(s_out))
        num_followers_d.append(len(d_in));  num_followees_d.append(len(d_out))
        inter_followers.append(len(s_in & d_in))
        inter_followees.append(len(s_out & d_out))

    return (num_followers_s, num_followees_s,
            num_followers_d, num_followees_d,
            inter_followers, inter_followees)


def compute_feature_stage2(df, graph, wcc, node_to_wcc_id, katz, hits_hub, hits_auth,
                            mean_katz, mean_hub, mean_auth):
    """Similarity, structural, and centrality features."""
    jac_fol, jac_fee   = [], []
    cos_fol, cos_fee   = [], []
    sp, swcc, adar, fb = [], [], [], []
    k_src, k_dst       = [], []
    h_src, h_dst       = [], []
    a_src, a_dst       = [], []

    for _, row in tqdm(df.iterrows(), total=len(df), desc='Stage-2'):
        src, dst = row['source_node'], row['destination_node']
        jac_fol.append(jaccard_for_followers(graph, src, dst))
        jac_fee.append(jaccard_for_followees(graph, src, dst))
        cos_fol.append(cosine_distance_for_followers(graph, src, dst))
        cos_fee.append(cosine_distance_for_followees(graph, src, dst))
        sp.append(compute_shortest_path_length(graph, src, dst))
        swcc.append(belongs_to_same_wcc(graph, wcc, node_to_wcc_id, src, dst))
        adar.append(calc_adar_in(graph, src, dst))
        fb.append(following_back(graph, src, dst))
        k_src.append(katz.get(src, mean_katz));  k_dst.append(katz.get(dst, mean_katz))
        h_src.append(hits_hub.get(src, mean_hub)); h_dst.append(hits_hub.get(dst, mean_hub))
        a_src.append(hits_auth.get(src, mean_auth)); a_dst.append(hits_auth.get(dst, mean_auth))

    return (jac_fol, jac_fee, cos_fol, cos_fee,
            sp, swcc, adar, fb,
            k_src, k_dst, h_src, h_dst, a_src, a_dst)

print('Feature computation functions defined.')

Feature computation functions defined.


In [11]:
%%time

mean_katz = sum(katz.values()) / len(katz)
mean_hub  = sum(hits_hub.values()) / len(hits_hub)
mean_auth = sum(hits_auth.values()) / len(hits_auth)

STAGE1_COLS = [
    'num_followers_s', 'num_followees_s',
    'num_followers_d', 'num_followees_d',
    'inter_followers', 'inter_followees',
]

print('Stage-1 — TRAIN ...')
(
    df_final_train['num_followers_s'], df_final_train['num_followees_s'],
    df_final_train['num_followers_d'], df_final_train['num_followees_d'],
    df_final_train['inter_followers'], df_final_train['inter_followees'],
) = compute_feature_stage1(df_final_train, train_graph)

print('Stage-1 — TEST ...')
(
    df_final_test['num_followers_s'], df_final_test['num_followees_s'],
    df_final_test['num_followers_d'], df_final_test['num_followees_d'],
    df_final_test['inter_followers'], df_final_test['inter_followees'],
) = compute_feature_stage1(df_final_test, train_graph)

print('\n Stage-1 done.')

Stage-1 — TRAIN ...


Stage-1:   0%|          | 0/99838 [00:00<?, ?it/s]

Stage-1 — TEST ...


Stage-1:   0%|          | 0/25056 [00:00<?, ?it/s]


 Stage-1 done.
CPU times: total: 3.92 s
Wall time: 4.05 s


In [12]:
%%time

STAGE2_COLS = [
    'jac_followers', 'jac_followees',
    'cos_followers', 'cos_followees',
    'shortest_path', 'same_wcc', 'adar_index', 'is_following_back',
    'katz_src', 'katz_dst',
    'hits_hub_src', 'hits_hub_dst',
    'hits_auth_src', 'hits_auth_dst',
]

print('Stage-2 — TRAIN ...')
(
    df_final_train['jac_followers'], df_final_train['jac_followees'],
    df_final_train['cos_followers'], df_final_train['cos_followees'],
    df_final_train['shortest_path'], df_final_train['same_wcc'],
    df_final_train['adar_index'],    df_final_train['is_following_back'],
    df_final_train['katz_src'],      df_final_train['katz_dst'],
    df_final_train['hits_hub_src'],  df_final_train['hits_hub_dst'],
    df_final_train['hits_auth_src'], df_final_train['hits_auth_dst'],
) = compute_feature_stage2(df_final_train, train_graph, wcc, node_to_wcc_id,
                            katz, hits_hub, hits_auth, mean_katz, mean_hub, mean_auth)

print('Stage-2 — TEST ...')
(
    df_final_test['jac_followers'], df_final_test['jac_followees'],
    df_final_test['cos_followers'], df_final_test['cos_followees'],
    df_final_test['shortest_path'], df_final_test['same_wcc'],
    df_final_test['adar_index'],    df_final_test['is_following_back'],
    df_final_test['katz_src'],      df_final_test['katz_dst'],
    df_final_test['hits_hub_src'],  df_final_test['hits_hub_dst'],
    df_final_test['hits_auth_src'], df_final_test['hits_auth_dst'],
) = compute_feature_stage2(df_final_test, train_graph, wcc, node_to_wcc_id,
                            katz, hits_hub, hits_auth, mean_katz, mean_hub, mean_auth)

print('\n Stage-2 done.')

Stage-2 — TRAIN ...


Stage-2:   0%|          | 0/99838 [00:00<?, ?it/s]

Stage-2 — TEST ...


Stage-2:   0%|          | 0/25056 [00:00<?, ?it/s]


 Stage-2 done.
CPU times: total: 1min 57s
Wall time: 2min 4s


In [13]:
os.makedirs('data', exist_ok=True)
df_final_train.to_csv('data/df_final_train_features.csv', index=False)
df_final_test.to_csv( 'data/df_final_test_features.csv',  index=False)
print('Feature DataFrames saved to data/')
df_final_train.head(3)

Feature DataFrames saved to data/


,source_node,destination_node,indicator_link,num_followers_s,num_followees_s,num_followers_d,num_followees_d,inter_followers,inter_followees,jac_followers,...,shortest_path,same_wcc,adar_index,is_following_back,katz_src,katz_dst,hits_hub_src,hits_hub_dst,hits_auth_src,hits_auth_dst
0,796395,866823,1,17,4,29,20,3,1,0.069767,...,2,1,0.286,0,0.000798,0.000845,1.589675e-16,8.064529e-14,2.232268e-15,5.259945e-15
1,1055622,788057,1,10,12,7,7,1,0,0.062500,...,2,1,0.000,1,0.000769,0.000759,1.186572e-16,1.599800e-16,2.092686e-15,2.651951e-15
2,573466,479767,0,1,0,0,1,0,0,0.000000,...,-1,-1,0.000,0,0.000735,0.000731,-0.000000e+00,3.719358e-17,3.537670e-19,-0.000000e+00
